# Rule: **build_industrial_production_per_country_tomorrow**


**Description**

The future industrial production per country is built using today's industrial productionand applies scenario-specific scaling factors only for the steel, alumina and HVC sectors, with parameters related with shares of the production processes from the configuration file according to the horizon year. However, the total production for these sectors remains intact. The remaining industrial sectors maintain their present production levels.

The configuration parameters that determine the future industrial production are defined under the **industry** section of the config file:  
- industry.St_primary_fraction
- industry.DRI_fraction
- industry.Al_primary_fraction
- industry.HVC_primary_fraction
- industry.HVC_mechanical_recycling_fraction
- industry.HVC_chemical_recycling_fraction

**Inputs**

- resources/{prefix}/{name}/`industrial_production_per_country.csv`

**Outputs**

- resources/{prefix}/{name}/`industrial_production_per_country_tomorrow_{horizon}.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

### Network
horizon = ''

In [ ]:
##### Imports
import pandas as pd
import os 
import sys
import matplotlib.pyplot as plt
import numpy as np

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### Set options
pd.set_option("display.max_columns", None)

## `industrial_production_per_country_tomorrow_{horizon}.csv`  
Load the file and preview its content.

In [ ]:
file = f"industrial_production_per_country_tomorrow_{horizon}.csv"

ind_prod_tomorrow = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

# Use the first column (country code) as DataFrame index
ind_prod_tomorrow = ind_prod_tomorrow.set_index(ind_prod_tomorrow.columns[0])

ind_prod_tomorrow.head()

See the future industrial production for a specific country

In [ ]:
# Select the country
country_code = "ES"

# Filter by country
ind_prod_tomorrow_country = ind_prod_tomorrow.loc[country_code]
ind_prod_tomorrow_country

What is the difference between current production and the expected production for the horizon year in the specified country?

Load present industrial production for comparison

In [ ]:
# Load the file
file = f"industrial_production_per_country.csv"

ind_prod_today = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

# Use the first column (country code) as DataFrame index
ind_prod_today = ind_prod_today.set_index(ind_prod_today.columns[0])

# Filter by country
ind_prod_today_country = ind_prod_today.loc[country_code]

See the industrial sectors undergoing process transformations

In [ ]:
# Layout for the graphs
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Colour dictionary
colours_subsectors = {
    # Steel
    "Electric arc": "#000dff",
    "Integrated steelworks": "#6ba4ee",
    "DRI + Electric arc": "#0ed3ff",

    # Aluminium
    "Aluminium - primary production": "#437812",
    "Aluminium - secondary production": "#8ee87b",

    # HVC
    "HVC": "#d62728",
    "HVC (mechanical recycling)": "#ed5fce",
    "HVC (chemical recycling)": "#975fb1",
}

today = pd.to_numeric(ind_prod_today_country, errors="coerce")
horizon_data = pd.to_numeric(ind_prod_tomorrow_country, errors="coerce")

today = today.drop("kton/a", errors="ignore")
horizon_data = horizon_data.drop("kton/a", errors="ignore")

########## Steel production
ax = axes[0]

steel_today_parts = ["Electric arc", "Integrated steelworks"]
steel_horizon_parts = ["Electric arc", "Integrated steelworks", "DRI + Electric arc"]

# Present production
bottom = 0
for part in steel_today_parts:
    val = today.get(part, 0)
    ax.bar(
        "today",
        val,
        bottom=bottom,
        label=part,
        color=colours_subsectors.get(part)
    )
    bottom += val

# Future production
bottom = 0
for part in steel_horizon_parts:
    val = horizon_data.get(part, 0)
    ax.bar(
        str(horizon),
        val,
        bottom=bottom,
        label=part,
        color=colours_subsectors.get(part)
    )
    bottom += val

ax.set_title("Steel")

######## Aluminium production
ax = axes[1]

al_parts = [
    "Aluminium - primary production",
    "Aluminium - secondary production"
]

# Present production
bottom = 0
for part in al_parts:
    val = today.get(part, 0)
    ax.bar(
        "today",
        val,
        bottom=bottom,
        label=part,
        color=colours_subsectors.get(part)
    )
    bottom += val

# Future production
bottom = 0
for part in al_parts:
    val = horizon_data.get(part, 0)
    ax.bar(
        str(horizon),
        val,
        bottom=bottom,
        color=colours_subsectors.get(part)
    )
    bottom += val

ax.set_title("Aluminium")


######## High value chemicals (HVC) production
ax = axes[2]

hvc_parts = [
    "HVC",
    "HVC (mechanical recycling)",
    "HVC (chemical recycling)"
]

# Present production
ax.bar(
    "today",
    today.get("HVC", 0),
    label="HVC",
    color=colours_subsectors.get("HVC")
)

# Future production
bottom = 0
for part in hvc_parts:
    val = horizon_data.get(part, 0)
    ax.bar(
        str(horizon),
        val,
        bottom=bottom,
        label=part,
        color=colours_subsectors.get(part)
    )
    bottom += val

ax.set_title("HVC")


######## Title and legend

fig.suptitle(f"Industrial sectors undergoing process transformations ({country_code})")

handles = []
labels = []

for ax in axes:
    h, l = ax.get_legend_handles_labels()
    for hi, li in zip(h, l):
        if li not in labels:
            handles.append(hi)
            labels.append(li)

fig.legend(
    handles,
    labels,
    loc="upper right",
    bbox_to_anchor=(1.2, 0.8),
    title="Subsectors"
)
axes[0].set_ylabel("Production (kton/a)")
plt.tight_layout(rect=[0.05, 0, 1, 1])
plt.show()

See the rest of the sectors

In [ ]:
# Exclude sectors that undergo transformations
exclude = set([
    "Electric arc",
    "Integrated steelworks",
    "DRI + Electric arc",
    "Aluminium - primary production",
    "Aluminium - secondary production",
    "HVC",
    "HVC (mechanical recycling)",
    "HVC (chemical recycling)"
])

# Sectors that do not undergo process transformations
other_sectors = [s for s in today.index if s not in exclude]

x = np.arange(len(other_sectors))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

today_vals = [today[s] for s in other_sectors]
horizon_vals = [horizon_data.get(s, 0) for s in other_sectors]

ax.bar(x - width/2, today_vals, width, label="Today")
ax.bar(x + width/2, horizon_vals, width, label=str(horizon))

ax.set_xticks(x)
ax.set_xticklabels(other_sectors, rotation=45, ha="right")

ax.set_title(f"Industrial sectors without transformation processes ({country_code})")
ax.legend()
ax.set_ylabel("Production (kton/a)")
plt.tight_layout()
plt.show()